In [2]:
import struct
import numpy as np
import math
from numpy.random import *
import WinoTran_NCHW as NCHW

In [3]:
#generation
chn = 512
numOfFilter =512
# print(224*224*64)
# print(112*112*128)
# print(56*56*256)
# print(28*28*512)
parameter = chn *3*3* numOfFilter

input1 = (np.array(rand(parameter))-0.5).astype(np.float32)
des = open("kernel.bin","wb")
cnt = des.write(input1)
des.close()

In [10]:
chn = 1
numOfFilter =64
inside = 224
bat4Conv =1
padding =1

inside_beta = math.ceil((inside+2*padding-2)/4)*4+2 
oside = inside +2*padding -2
blockn = (int)((inside_beta-2)/4)
M = bat4Conv * blockn*blockn
N = numOfFilter;
K = chn
MSize = M if (M%128 == 0) else math.ceil(M/128)*128
NSize = N if (N%128 == 0) else math.ceil(N/128)*128
KSize = (int((K-1)/8)+1)*8

parameters1 = 36*MSize*NSize
print(MSize, NSize, parameters1)

# readin the feature map
src = open("../../M3/data/gemmOut.bin","rb")
context = src.read(parameters1*4)
real_context = struct.unpack(str(parameters1)+'f',context)
input = np.array(real_context)
# input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
sample_gemmout = input.reshape((36,MSize,NSize)).astype(np.float32)
print(sample_gemmout.shape)

gemmTran = NCHW.Wino_OutputTran(sample_gemmout,oside,bat4Conv,numOfFilter)
finalOutput = NCHW.Wino_inverseTran(gemmTran, numOfFilter, bat4Conv, blockn, oside)
print(finalOutput.shape)

# convOutput = NCHW.Wino_inverseTran(sample_gemmout, )(outputTran,chn, batch, blockn,oside):


3200 128 14745600
(36, 3200, 128)
(1, 64, 224, 224)


In [34]:
for i in range(0,1):
    parameter2 = bat4Conv* oside * oside* numOfFilter
    src = open("./M4_new"+str(i)+".bin","rb")
    context = src.read(parameter2*4)
    real_context = struct.unpack(str(parameter2)+'f',context)
    input = np.array(real_context)
    # input = input.reshape((chn,inside,inside,batch)).astype(np.float32).transpose((3,1,2,0))
    testoutput = input.reshape((bat4Conv, numOfFilter, oside , oside)).astype(np.float32)
    print( np.sum(np.abs(finalOutput[:,:,:]-testoutput[:,:,:])) )

0.12349935


In [35]:
err = finalOutput - testoutput
print(err.shape)
print(np.sum(np.abs(err)))

(1, 64, 224, 224)
0.12349935


In [ ]:
print(err[0,0,])

In [32]:
print(finalOutput.shape)
print(finalOutput[0,0,0:8,0:8])

(1, 64, 224, 224)
[[ 0.1418761   0.04582218 -0.5014222  -0.04562296  0.48448148  0.03217798
  -0.32455027  0.02914651]
 [-0.08817179 -0.3309452   0.5940781   0.17303716 -0.33637598 -0.15035616
   0.04141303  0.27776736]
 [ 0.20983198 -0.08506608 -0.11092955 -0.02529889 -0.09608671 -0.1724046
   0.25720224 -0.2508165 ]
 [-0.07286861  0.08421125 -0.18054941  0.09822639  0.20117922 -0.08758833
   0.30575845  0.43094772]
 [-0.29254347 -0.11645909  0.28047454  0.22672068 -0.06854463  0.34504637
  -0.6478696  -0.05664694]
 [ 0.5171727  -0.35928658 -0.10611917 -0.131513   -0.01415369  0.11898417
   0.64623743 -0.4919085 ]
 [-0.1684694   0.60190254 -0.46905932  0.09385531  0.09013522 -0.06869353
  -0.13179176  0.17533843]
 [-0.43563762  0.057074    0.09568889  0.35298085 -0.74447125  0.4475637
  -0.40636528 -0.16272277]]


In [31]:
print(testoutput[0,0,0:9,0:9])

[[ 0.14187615  0.04582214 -0.5014222  -0.04562294  0.48448163  0.03217781
  -0.32455015  0.02914619  0.0158206 ]
 [ 0.          0.         -0.08817175 -0.33094522  0.5940782   0.17303705
  -0.33637592 -0.15035623  0.04141313]
 [-0.57953215  0.09553349  0.          0.          0.20983203 -0.08506611
  -0.11092955 -0.02529871 -0.09608667]
 [ 0.6157467  -0.9103391   0.40026355 -0.00550067  0.          0.
  -0.07286859  0.08421122 -0.18054935]
 [-0.08880463 -0.22707963 -0.06284083  0.6508907  -0.36444145 -0.09063935
   0.          0.         -0.29254344]
 [ 0.69228536 -0.41968253 -0.19010073  0.54015654 -0.49723434 -0.02850246
   0.20377731  0.12648344  0.        ]
 [-0.68306005  0.5063546  -0.67026985  0.56074303 -0.01406139 -0.60360587
   0.7861501  -0.56280726 -0.03467572]
 [ 0.16055699 -0.09776464  0.6261771  -0.2044012   0.45993978 -0.7099809
   0.46051612  0.03243387 -0.6521073 ]
 [ 0.1113188  -0.14892218  0.1881389  -0.02993593 -0.07709253  0.46173686
  -0.5476842   0.26772    -0.37